In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

def run_xgb_with_pca(
    X_train, y_train,
    X_test, y_test=None,
    variance_threshold=0.95,
    n_estimators=800,
    max_depth=7,
    learning_rate=0.01,
    random_state=42
):
    """
    Train and evaluate XGBoost regressor on PCA-transformed features.
    """

    # -----------------------------
    # 1. Standardize inputs
    # -----------------------------
    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled  = scaler_X.transform(X_test)

    # -----------------------------
    # 2. PCA
    # -----------------------------
    pca = PCA(n_components=variance_threshold)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca  = pca.transform(X_test_scaled)

    # PCA loadings for reference
    pca_loadings = pd.DataFrame(
        pca.components_.T,
        columns=[f"PC{i+1}" for i in range(pca.n_components_)],
        index=[f"feat_{i}" for i in range(X_train.shape[1])]
    )

    # -----------------------------
    # 3. Train XGBoost
    # -----------------------------
    model = xgb.XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        objective='reg:squarederror',
        random_state=random_state,
        verbosity=1
    )
    model.fit(X_train_pca, y_train)

    # -----------------------------
    # 4. Predict
    # -----------------------------
    y_pred = model.predict(X_test_pca)

    # -----------------------------
    # 5. Metrics
    # -----------------------------
    metrics = {}
    if y_test is not None:
        metrics['MSE'] = mean_squared_error(y_test, y_pred)
        metrics['RMSE'] = np.sqrt(metrics['MSE'])
        metrics['MAE'] = mean_absolute_error(y_test, y_pred)
        metrics['R2'] = r2_score(y_test, y_pred)


    # -----------------------------
    # 6. Plot predictions vs true
    # -----------------------------
    plt.figure(figsize=(14,6))

    # Predicted line
    plt.plot(y_pred, label="Predicted (line)", linewidth=2, color='blue')

    # Predicted points (optional, same as line)
    plt.scatter(np.arange(len(y_pred)), y_pred, label="Predicted (points)", color='blue', alpha=0.5, s=30)

    # True values as points
    if y_test is not None:
        plt.scatter(np.arange(len(y_test)), y_test, label="True Values", color='black', s=50)

    plt.title("XGBoost Predictions (PCA Features)")
    plt.xlabel("Test Sample Index")
    plt.ylabel("Target")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return {
        'y_pred': y_pred,
        'model': model,
        'scaler_X': scaler_X,
        'pca': pca,
        'pca_loadings': pca_loadings,
        'metrics': metrics,
        'X_train_pca': X_train_pca,
        'X_test_pca': X_test_pca
    }


def preprocess_and_split(df, feature_cols, target_col, test_frac=0.20):
    """
    Processes df into event-level stats, filters NA, 
    sorts by target ascending, and splits bottom % as test set.

    Returns:
        X_train, X_test, y_train, y_test, df_filt
    """

    # ----------------------------------------------------
    # 1. Build per-event aggregation
    # ----------------------------------------------------
    df_event_stats = (
        df.groupby("date")[
            [
                "choice_ev", "choice_proba", 
                "choice_fstar", "choice_accuracy",
                "choice_decimal_odds", "choice_bet_sharpe",
                "choice_bet_sigma", "choice_juice",
                "sharpe_portfolio", "portfolio_sigma",
                "sigma_portfolio_scaled", 'choice_net_odds', 'choice_correct', 'choice_incorrect'
            ]
        ]
        .mean()
        .reset_index()
    )


    # Extra per-event features
    df_event_stats["total_fights"] = df.groupby("date").size().values
    df_event_stats["sum_fstar"] = df.groupby("date")["choice_fstar"].sum().values
    
    df_event_stats["f_net"] = df_event_stats["choice_net_odds"] * df_event_stats["choice_fstar"]
    df_event_stats["f_event_net"] = df_event_stats.groupby("date")["f_net"].transform("sum")

    new_cols = ['sum_fstar', 'total_fights', 'choice_correct', 'choice_incorrect']
    all_feats = feature_cols + new_cols
    # ----------------------------------------------------
    # 2. Drop NA rows that cannot be used
    # ----------------------------------------------------
    df_filt = df_event_stats.dropna(subset=all_feats + [target_col])

    # ----------------------------------------------------
    # 3. Sort by target ASCENDING (test = bottom 20%)
    # ----------------------------------------------------
    df_sorted = df_filt.sort_values('date', ascending=True).reset_index(drop=True)

    n = len(df_sorted)
    n_test = int(n * test_frac)

    df_test = df_sorted.iloc[:n_test]
    df_train = df_sorted.iloc[n_test:]

    # ----------------------------------------------------
    # 4. Extract X and y
    # ----------------------------------------------------
    X_train = df_train[all_feats].values
    y_train = df_train[target_col].values

    X_test  = df_test[all_feats].values
    y_test  = df_test[target_col].values

    return X_train, X_test, y_train, y_test, df_filt


def preprocess_and_split_per_bet(df, feature_cols, target_col, test_frac=0.20):
    """
    Processes df per bet (no grouping).
    Filters NA, sorts by date ascending, and splits bottom % as test set.

    Returns:
        X_train, X_test, y_train, y_test, df_filtered
    """

    # ---------------------------------------------
    # 1. Add additional features if they exist
    # ---------------------------------------------
    extra_cols = []

    # Add per-bet net outcome
    if "choice_net_odds" in df.columns and "choice_fstar" in df.columns:
        df["f_net"] = df["choice_net_odds"] * df["choice_fstar"]
        extra_cols.append("f_net")

    # You referenced these — keep them if they exist
    for col in ['choice_correct', 'choice_incorrect']:
        if col in df.columns:
            extra_cols.append(col)

    # All passthrough features
    all_feats = feature_cols + extra_cols

    # ---------------------------------------------
    # 2. Filter NA on the selected features + target
    # ---------------------------------------------
    df_filtered = df.dropna(subset=all_feats + [target_col]).copy()

    # ---------------------------------------------
    # 3. Sort by date ASCENDING
    # ---------------------------------------------
    df_sorted = df_filtered.sort_values("date", ascending=True).reset_index(drop=True)

    # ---------------------------------------------
    # 4. Bottom % of samples → test set
    # ---------------------------------------------
    n = len(df_sorted)
    n_test = int(n * test_frac)

    df_test  = df_sorted.iloc[:n_test]
    df_train = df_sorted.iloc[n_test:]

    # ---------------------------------------------
    # 5. Extract matrices
    # ---------------------------------------------
    X_train = df_train[all_feats].values
    y_train = df_train[target_col].values

    X_test = df_test[all_feats].values
    y_test = df_test[target_col].values

    return X_train, X_test, y_train, y_test, df_filtered




In [ ]:
feature_cols = [
    "choice_ev", "choice_proba", "choice_fstar",
    "choice_accuracy", "choice_decimal_odds", "choice_bet_sharpe",
    "choice_bet_sigma", "choice_juice", 'choice_correct', 'choice_incorrect', 'choice_accuracy'
    ]



X_train, X_test, y_train, y_test, df_filt = preprocess_and_split_per_bet(
    df_kelly,
    feature_cols,
    target_col="choice_net_odds"   # or whatever your target is
)

results = run_xgb_with_pca(X_train, y_train, X_test, y_test)

In [ ]:
feature_cols = [
    "choice_ev", "choice_proba", "choice_fstar",
    "choice_accuracy", "choice_decimal_odds", "choice_bet_sharpe",
    "choice_bet_sigma", "choice_juice", "sharpe_portfolio",
    "portfolio_sigma", "sigma_portfolio_scaled"
]

df_kelly["choice_net_odds"] = df_kelly["choice_decimal_odds"] - 1
df_kelly["choice_net_odds"] = np.where(df_kelly['winner']==df_kelly['pred_winner'], df_kelly["choice_net_odds"], -1*df_kelly["choice_net_odds"])

target_col = "f_event_net"
target_col = "choice_net_odds"

X_train, X_test, y_train, y_test, df_filt = preprocess_and_split(
    df_kelly,
    feature_cols,
    target_col,
    test_frac=0.20   # bottom 20%
)


results_dict = run_xgb_with_pca(X_train, y_train, X_test, y_test, n_estimators=500,
    max_depth=5,
    learning_rate=0.005)


In [ ]:
from sklearn.ensemble import RandomForestRegressor


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# -------------------------
# Utilities
# -------------------------
def pinball_loss(y_true, y_pred, q):
    """
    Pinball (quantile) loss for quantile q (0<q<1).
    """
    e = y_true - y_pred
    return np.mean(np.maximum(q * e, (q - 1) * e))


# -------------------------
# 1) Train quantile LightGBM models
# -------------------------
def train_lgb_quantiles_pre_split(
    X_train, y_train,
    X_test, y_test, feature_cols,
    quantiles=(0.05, 0.5, 0.95),
    num_boost_round=2000,
    early_stopping_rounds=50,
    verbose_eval=200,
):
    models = {}
    best_iters = {}

    # Standardize numerics (important for boosting stability)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    dtrain = lgb.Dataset(X_train_s, label=y_train)
    dval   = lgb.Dataset(X_test_s, label=y_test)

    for q in quantiles:
        params_q = {
            "objective": "quantile",
            "alpha": q,
            "learning_rate": 0.03,
            "num_leaves": 64,
            "min_data_in_leaf": 20,
            "feature_fraction": 0.9,
            "bagging_fraction": 0.9,
            "bagging_freq": 1,
            "metric": "quantile",
            "verbosity": -1,
        }

        print(f"\n--- Training quantile model q={q} ---")

        model = lgb.train(
            params_q,
            dtrain,
            num_boost_round=num_boost_round,
            valid_sets=[dtrain, dval],
            callbacks=[
                lgb.early_stopping(stopping_rounds=early_stopping_rounds),
                lgb.log_evaluation(verbose_eval),
            ]
        )

        models[q] = model
        best_iters[q] = model.best_iteration

    return {
        "models": models,
        "scaler": scaler,
        "best_iters": best_iters,
        "X_test": X_test,
        "y_test": y_test,
        "quantiles": quantiles,
        "feature_cols": feature_cols
    }


# -------------------------
# 2) Predict function
# -------------------------
def predict_lgb_quantiles(models_dict, X_raw):
    """
    Predict quantiles for rows in X_raw (unscaled). 
    models_dict: result of train_lgb_quantiles
    X_raw: numpy array or DataFrame of raw features (not scaled)
    Returns: DataFrame with columns q_{quantile}
    """
    scaler = models_dict["scaler"]
    models = models_dict["models"]
    feature_cols = models_dict["feature_cols"]

    X_np = X_raw.values if isinstance(X_raw, pd.DataFrame) else np.asarray(X_raw)
    Xs = scaler.transform(X_np)

    preds = {}
    for q, model in models.items():
        preds[q] = model.predict(Xs, num_iteration=models_dict["best_iters"][q])

    df_out = pd.DataFrame(preds)
    # reorder columns (from smallest to largest quantile)
    df_out = df_out.reindex(sorted(df_out.columns), axis=1)
    # rename columns to q05, q50, q95 style
    df_out.columns = [f"q_{int(q*100):02d}" for q in df_out.columns]
    return df_out


# -------------------------
# 3) Evaluate models: pinball loss and coverage
# -------------------------
def evaluate_quantile_models(models_dict):
    """
    Evaluate pinball loss per quantile and empirical coverage of interval [q05,q95].
    """
    X_test = models_dict["X_test"]
    y_test = models_dict["y_test"]
    models = models_dict["models"]
    best_iters = models_dict["best_iters"]

    preds = {}
    for q, model in models.items():
        preds[q] = model.predict(X_test, num_iteration=best_iters[q])

    # Compute pinball loss
    pinball = {q: pinball_loss(y_test, preds[q], q) for q in preds}

    # coverage of [0.05, 0.95]
    lower = preds[min(preds.keys())]
    upper = preds[max(preds.keys())]
    coverage = np.mean((y_test >= lower) & (y_test <= upper))

    # Median MAE
    if 0.5 in preds:
        mae_median = mean_absolute_error(y_test, preds[0.5])
    else:
        mae_median = None

    return {
        "pinball": pinball,
        "coverage_05_95": coverage,
        "mae_median": mae_median,
        "y_test": y_test,
        "preds": preds
    }


# -------------------------
# 4) Plot predictions vs actuals with intervals
# -------------------------
def plot_predictions_with_intervals(models_dict, n_plot=100):
    eval_res = evaluate_quantile_models(models_dict)
    y_test = eval_res["y_test"]
    preds = eval_res["preds"]

    # select indices to plot
    m = len(y_test)
    idx = np.arange(min(n_plot, m))

    lower = preds[min(preds.keys())][idx]
    median = preds[0.5][idx]
    upper = preds[max(preds.keys())][idx]
    y = y_test[idx]

    plt.figure(figsize=(12,5))
    plt.plot(idx, y, "o", label="actual")
    plt.plot(idx, median, "r-", label="pred median")
    plt.fill_between(idx, lower, upper, color="orange", alpha=0.25, label="90% interval")
    plt.legend()
    plt.xlabel("Test sample index")
    plt.ylabel("event_net_odds")
    plt.title("Predictions (median) and 90% intervals vs actuals")
    plt.show()





In [ ]:
feature_cols = [
    "choice_ev", "choice_proba", "choice_fstar",
    "choice_decimal_odds", "choice_bet_sharpe", "choice_bet_sigma"
]
# --- Chronological split 80/20 ---
df_filtered = df_kelly[(df_kelly['choice_ev'] > 0) & 
                       (df_kelly['choice_fstar'] > 0)].copy()

n = len(df_filtered)
split_idx = int(n * 0.8)

df_train = df_filtered.iloc[:split_idx].copy()
df_test  = df_filtered.iloc[split_idx:].copy()

print("Train size:", len(df_train))
print("Test size:", len(df_test))
# optional filtering (your usual filter)

X_train = df_train[feature_cols].values
y_train = df_train["event_net_odds"].values

X_test  = df_test[feature_cols].values
y_test  = df_test["event_net_odds"].values

models_dict = train_lgb_quantiles_pre_split(
    X_train, y_train,
    X_test, y_test,
    feature_cols=feature_cols,
    quantiles=(0.05, 0.5, 0.95)
)

eval_res = evaluate_quantile_models(models_dict)

print("Pinball losses:", eval_res["pinball"])
print("Coverage (5–95):", eval_res["coverage_05_95"])
print("Median MAE:", eval_res["mae_median"])
# Quick plot
plot_predictions_with_intervals(models_dict, n_plot=80)

df_test_preds = predict_lgb_quantiles(models_dict, df_test[feature_cols])

df_test = df_test.reset_index(drop=True)
df_test = pd.concat([df_test, df_test_preds], axis=1)

df_test.head()

In [ ]:
def plot_lower_quantile_vs_actual(models_dict):
    """
    Scatter plot: lower quantile (q05) on x-axis vs actual event_net_odds on y-axis.
    """
    import matplotlib.pyplot as plt

    eval_res = evaluate_quantile_models(models_dict)
    y_test = eval_res["y_test"]
    preds = eval_res["preds"]

    # lower quantile
    lower = preds[min(preds.keys())]  # corresponds to q05
    upper = preds[max(preds.keys())] 

    lower = np.abs(upper) - np.abs(lower)
    plt.figure(figsize=(8,6))
    plt.scatter(lower, y_test, alpha=0.6)
    plt.axvline(0, color='red', linestyle='--', label='q05=0')
    plt.axhline(0, color='blue', linestyle='--', label='actual=0')
    plt.xlabel("Predicted lower quantile (q05)")
    plt.ylabel("Actual event_net_odds")
    plt.title("Lower Quantile vs Actual Event Net Odds")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
pred_df = predict_lgb_quantiles(models_dict, X_test)  # returns DataFrame with q05, q50, q95

# Count how many events have negative lower quantiles
n_neg_lower = (pred_df['q_05'] < 0).sum()
total_events = len(pred_df)
print(f"{n_neg_lower}/{total_events} events have negative lower quantile")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import brier_score_loss

def prepare_event_binomial_df(df_results, filter_choice_positive=True):
    """
    Build per-event (per-date) binomial dataset.
    - df_results: raw per-fight DataFrame (must include date, pred_winner, winner, choice_fstar, etc.)
    - filter_choice_positive: if True, only include fights with choice_ev > 0 and choice_fstar > 0, matching your earlier filter.
    Returns: df_events with columns:
      date, total_fights (n), correct (k), observed_prop (=k/n), and aggregated predictor columns.
    """
    df = df_results.copy()
    if filter_choice_positive:
        df = df[(df['choice_ev'] > 0) & (df['choice_fstar'] > 0)].copy()

    # correct (0/1) per fight
    df['correct'] = (df['pred_winner'] == df['winner']).astype(int)

    # Group by event date and compute:
    # n = count of fights, k = sum(correct), observed proportion = k/n
    agg_funcs = {
        'correct': ['sum', 'count'],                # sum -> k, count -> n
        'choice_fstar': ['mean', 'sum'],
        'choice_ev': 'mean',
        'choice_proba': 'mean',
        'event_net_odds': 'mean',
        'choice_decimal_odds': 'mean',
        'choice_bet_sharpe': 'mean',
        'choice_bet_sigma': 'mean'
    }

    df_event = df.groupby('date').agg(agg_funcs)
    # flatten multiindex columns
    df_event.columns = ['_'.join(col).strip() for col in df_event.columns.values]

    # rename and compute observed proportion
    df_event = df_event.rename(columns={
        'correct_sum': 'k_correct',
        'correct_count': 'n_fights',
        'choice_fstar_mean': 'choice_fstar_mean',
        'choice_fstar_sum': 'choice_fstar_sum',
        'choice_ev_mean': 'choice_ev_mean'
    })

    df_event['observed_prop'] = df_event['k_correct'] / df_event['n_fights']

    # reset index
    df_event = df_event.reset_index()

    # drop any rows where n_fights == 0 or NaNs
    df_event = df_event[df_event['n_fights'] > 0].dropna().reset_index(drop=True)

    return df_event


def fit_binomial_glm(df_event, predictors, train_frac=0.8, random_state=0, chrono_split=True):
    """
    Fit Binomial GLM on per-event aggregated data.
    - df_event: output of prepare_event_binomial_df
    - predictors: list of column names to use as covariates (these should exist in df_event)
    - train_frac: fraction for training (chronological if chrono_split True)
    Returns: dict with model, data splits, predictions, metrics, and plots shown.
    """

    # Ensure correct dtypes
    df_event = df_event.copy()
    df_event = df_event.sort_values('date').reset_index(drop=True)  # chronological

    # Build design matrix X and response k, n
    required = ['k_correct', 'n_fights']
    for r in required:
        if r not in df_event.columns:
            raise ValueError(f"Missing required column {r} in df_event")

    # split train/test chronologically
    if chrono_split:
        n = len(df_event)
        split_idx = int(np.floor(n * train_frac))
        df_train = df_event.iloc[:split_idx].copy().reset_index(drop=True)
        df_test  = df_event.iloc[split_idx:].copy().reset_index(drop=True)
    else:
        df_train, df_test = train_test_split(df_event, train_size=train_frac, random_state=random_state, shuffle=True)

    # Build formula for GLM: we will use statsmodels' GLM with family Binomial and freq_weights = n
    # The response will be df_train['k_correct'] / df_train['n_fights'] with freq_weights = n_fights
    formula = "observed_prop ~ " + " + ".join(predictors)

    # fit on train
    model = smf.glm(
        formula=formula,
        data=df_train,
        family=sm.families.Binomial(),
        freq_weights=df_train['n_fights']  # weight each event by number of fights (trials)
    ).fit()

    print("\n=== GLM (Binomial family, logit link) summary (TRAIN) ===")
    print(model.summary())

    # Odds ratios and CI
    params = model.params
    conf = model.conf_int()
    or_df = pd.DataFrame({
        'coef': params,
        'odds_ratio': np.exp(params),
        'ci_lower': conf[0],
        'ci_upper': conf[1],
        'p_value': model.pvalues
    })
    or_df['or_ci_lower'] = np.exp(or_df['ci_lower'])
    or_df['or_ci_upper'] = np.exp(or_df['ci_upper'])

    print("\n=== Odds Ratios (exp(coef)) and 95% CI ===")
    print(or_df[['coef','odds_ratio','or_ci_lower','or_ci_upper','p_value']])

    # Marginal effects (average marginal effects)
    try:
        mfx = model.get_margeff(at='overall', method='dydx')
        print("\n=== Marginal Effects (average) ===")
        print(mfx.summary())
    except Exception as e:
        print("Could not compute marginal effects:", e)
        mfx = None

    # Predictions: obtain predicted probability p_hat for train and test
    # Use model.predict on full exog; it returns predicted mean (probability) on response scale
    df_train['pred_p'] = model.predict(df_train)
    df_test['pred_p']  = model.predict(df_test)

    # Weighted Brier score: sum_n (p - k/n)^2 * n / sum(n) = sum((p*n - k)^2) / sum(n)
    def weighted_brier(df_):
        p = df_['pred_p'].values
        k = df_['k_correct'].values
        n = df_['n_fights'].values
        return np.sum((p * n - k) ** 2) / np.sum(n)

    train_brier = weighted_brier(df_train)
    test_brier  = weighted_brier(df_test)

    # Calibration: observed proportion vs predicted probability (binned)
    def calibration_plot(df_, title="Calibration"):
        bins = np.linspace(0, 1, 6)  # 5 bins
        df_['p_bin'] = pd.cut(df_['pred_p'], bins=bins, include_lowest=True)
        calib = df_.groupby('p_bin').apply(lambda g: pd.Series({
            'pred_mean': g['pred_p'].mean(),
            'obs_prop': g['k_correct'].sum() / g['n_fights'].sum(),
            'n_events': len(g),
            'n_trials': g['n_fights'].sum()
        })).reset_index()
        # plot
        plt.figure(figsize=(6,5))
        plt.plot([0,1],[0,1], 'k--', alpha=0.6)
        plt.scatter(calib['pred_mean'], calib['obs_prop'], s=20 + calib['n_trials']/2, alpha=0.8)
        for _, r in calib.iterrows():
            plt.text(r['pred_mean'], r['obs_prop'], f"n={r['n_trials']}", fontsize=8, va='bottom')
        plt.xlabel('Predicted probability')
        plt.ylabel('Observed proportion (k/n)')
        plt.title(title)
        plt.grid(alpha=0.3)
        plt.show()
        return calib

    # Pred vs observed scatter (per-event)
    def pred_vs_obs_scatter(df_, title="Pred vs Observed"):
        plt.figure(figsize=(7,5))
        plt.scatter(df_['pred_p'], df_['observed_prop'], alpha=0.7)
        plt.plot([0,1],[0,1], 'k--', alpha=0.6)
        plt.xlabel("Predicted probability (per-fight)")
        plt.ylabel("Observed proportion (k/n)")
        plt.title(title)
        plt.grid(alpha=0.3)
        plt.show()

    # Plots
    print("\n--- Calibration (TRAIN) ---")
    calib_train = calibration_plot(df_train, title="Calibration (Train)")
    pred_vs_obs_scatter(df_train, title="Predicted vs Observed (Train)")

    print("\n--- Calibration (TEST) ---")
    calib_test = calibration_plot(df_test, title="Calibration (Test)")
    pred_vs_obs_scatter(df_test, title="Predicted vs Observed (Test)")

    # Compute predicted expected correct counts; compare aggregate
    total_k_train = df_train['k_correct'].sum()
    total_k_train_pred = (df_train['pred_p'] * df_train['n_fights']).sum()

    total_k_test = df_test['k_correct'].sum()
    total_k_test_pred = (df_test['pred_p'] * df_test['n_fights']).sum()

    print(f"\nTrain: observed total correct = {total_k_train}, predicted total correct = {total_k_train_pred:.2f}")
    print(f"Test : observed total correct = {total_k_test}, predicted total correct = {total_k_test_pred:.2f}")

    results = {
        'model': model,
        'mfx': mfx,
        'or_table': or_df,
        'df_train': df_train,
        'df_test': df_test,
        'train_brier': train_brier,
        'test_brier': test_brier,
        'calib_train': calib_train,
        'calib_test': calib_test
    }

    return results


# 1) Build events dataset
df_event = prepare_event_binomial_df(df_kelly, filter_choice_positive=True)

# 2) Choose predictors (must be present in df_event). For example:
predictors = [ 'n_fights','choice_proba_mean']

# 3) Fit & evaluate
res = fit_binomial_glm(df_event, predictors, train_frac=0.8)

# Inspect results
print("Train Brier:", res['train_brier'])
print("Test  Brier:", res['test_brier'])
print(res['or_table'])        # odds ratios and p-values
print(res['model'].summary()) # full GLM summary


In [ ]:
import statsmodels.api as sm
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


def run_linear_model_with_pca(
        df,
        feature_cols,
        target_col="event_net_odds",
        variance_threshold=0.95
    ):
    """
    1. Applies event-level filtering
    2. Standardizes features
    3. Runs PCA to retain 95% variance
    4. Runs OLS on PCA components
    5. Runs WLS to correct heteroskedasticity
    6. Runs Quantile Regression (median)
    7. Shows a residual plot
    """

    # ----------------------------------------------------
    # 1. MATCH FILTERING USED EARLIER
    # ----------------------------------------------------
    df_event_stats = (
        df.groupby("date")[
            [
                "event_net_odds", "choice_ev", "choice_proba", "choice_fstar", 'choice_accuracy',
                "choice_decimal_odds", "choice_bet_sharpe", "choice_bet_sigma", 'choice_juice', 'sharpe_portfolio', 'portfolio_sigma', 'sigma_portfolio_scaled'
            ]
        ]
        .mean()
        .reset_index()
    )
    df_event_stats["total_fights"] = df.groupby("date").size().values
    df_event_stats["sum_fstar"] = df.groupby("date")["choice_fstar"].sum().values
    df_event_stats['choice_net_odds'] = df['choice_decimal_odds']-1
    df_event_stats['f_net'] = df_event_stats['choice_net_odds'] * df_event_stats['choice_fstar']
    df_event_stats['f_event_net'] = df_event_stats.groupby("date")["f_net"].transform("sum")

    df_filt = df_event_stats.dropna(subset=feature_cols + [target_col])

    # ----------------------------------------------------
    # 2. FEATURE MATRIX + TARGET
    # ----------------------------------------------------
    X_raw = df_filt[feature_cols].values
    y = df_filt[target_col].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)

    # ----------------------------------------------------
    # 3. PCA (retain 95% variance)
    # ----------------------------------------------------
    pca = PCA(n_components=variance_threshold)
    X_pca = pca.fit_transform(X_scaled)

    print(f"\n📌 PCA components retained: {X_pca.shape[1]}")
    print(f"📌 Variance retained: {pca.explained_variance_ratio_.sum():.4f}")

    # Add constant term
    X_pca_const = sm.add_constant(X_pca)

    # ----------------------------------------------------
    # 4. OLS Fit
    # ----------------------------------------------------
    model = sm.OLS(y, X_pca_const).fit()

    print("\n================= OLS SUMMARY =================")
    print(model.summary())
    residuals = model.resid
    fitted = model.fittedvalues

    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=fitted, y=residuals, alpha=0.6)
    plt.axhline(0, color='red', linestyle='--')
    plt.xlabel("Fitted Values")
    plt.ylabel("Residuals")
    plt.title("Residuals vs Fitted Values")
    plt.tight_layout()
    plt.show()

    
    plt.figure(figsize=(8, 8))
    sm.qqplot(residuals, line='45', fit=True)
    plt.title("Q-Q Plot of Residuals")
    plt.tight_layout()
    plt.show()

    sm.graphics.influence_plot(model, figsize=(10, 8))
    plt.title("Influence Plot")
    plt.show()

    from sklearn.preprocessing import PolynomialFeatures

    poly = PolynomialFeatures(degree=2, include_bias=False)
    X_poly = poly.fit_transform(X_pca_const)

    model_poly = sm.OLS(y, X_poly).fit()
    print(model_poly.summary())

    fitted = model_poly.fittedvalues
    residuals = model_poly.resid

    plt.figure(figsize=(10,6))
    plt.scatter(fitted, residuals, alpha=0.6)
    plt.axhline(0, color="red", linestyle="--", linewidth=2)
    plt.xlabel("Fitted Values")
    plt.ylabel("Residuals")
    plt.title("Polynomial Regression Residuals vs Fitted Values")
    plt.show()
    # ----------------------------------------------------
    # 5. WLS Fit (heteroskedasticity correction)
    # ----------------------------------------------------
    weights = 1 / np.maximum(model.fittedvalues ** 2, 1e-8)
    wls_model = sm.WLS(y, X_pca_const, weights=weights).fit()
    residuals = wls_model.resid
    plt.figure(figsize=(8,5))
    plt.scatter(wls_model.fittedvalues, residuals, alpha=0.6)
    plt.axhline(0, color='red', linestyle='--')
    plt.xlabel("Fitted Values")
    plt.ylabel("Residuals")
    plt.title("WLS Residuals vs Fitted Values")
    plt.show()


    # Standardized residuals
    y_hat = wls_model.fittedvalues

    # Standard error of prediction
    pred_se = wls_model.get_prediction(X_pca_const).se_mean  # SE of the mean prediction
    pred_int = wls_model.get_prediction(X_pca_const).conf_int(alpha=0.05)  # 95% CI for mean

    # Quantiles: 2.5% and 97.5% for each observation
    lower, upper = pred_int[:,0], pred_int[:,1]

    # Combine in a DataFrame
    import pandas as pd
    df_quantiles = pd.DataFrame({
        'y_hat': y_hat,
        'lower_2.5%': lower,
        'upper_97.5%': upper
    })

    print(df_quantiles.head())
    pred_obs = wls_model.get_prediction(X_pca_const).summary_frame(alpha=0.05)

    print("\n================= WLS SUMMARY =================")
    print(wls_model.summary())

    # ----------------------------------------------------
    # 6. Quantile Regression (Median)
    # ----------------------------------------------------
    # QR uses *raw filtered data*, not PCA representation
    formula = target_col + " ~ " + " + ".join(feature_cols)
    qr_model = smf.quantreg(formula, df_filt)
    qr_med = qr_model.fit(q=0.5)

    print("\n================= Quantile Regression (Median) =================")
    print(qr_med.summary())

    # ----------------------------------------------------
    # 7. Residual Plot (after OLS)
    # ----------------------------------------------------


    return {
        "ols_model": model,
        "wls_model": wls_model,
        "qr_model_median": qr_med,
        "scaler": scaler,
        "pca": pca,
        "feature_cols": feature_cols,
        "target_col": target_col,
        "df_used": df_filt,
    }

def pca_linear_predict(pipeline, new_event, alpha=0.05):
    scaler = pipeline["scaler"]
    pca = pipeline["pca"]
    model = pipeline["model"]
    feature_cols = pipeline["feature_cols"]

    # Convert new event to row vector
    X_new = pd.DataFrame([new_event])[feature_cols].values

    # Standardize
    X_scaled = scaler.transform(X_new)

    # PCA transform
    X_pca = pca.transform(X_scaled)

    # Add constant for OLS prediction
    X_pca_const = sm.add_constant(X_pca, has_constant="add")

    pred = model.get_prediction(X_pca_const)

    return {
        "mean_pred": float(pred.predicted_mean[0]),
        "conf_int": tuple(pred.conf_int(alpha=alpha)[0]),
        "pred_int": tuple(pred.conf_int(obs=True, alpha=alpha)[0]),
    }

def plot_pca_loadings(pca, feature_names, n_components=5):
    """
    Plots the loadings of the first n PCA components.
    """
    components = min(n_components, pca.components_.shape[0])
    loadings = pca.components_[:components]

    plt.figure(figsize=(10, 6))
    im = plt.imshow(loadings, cmap="coolwarm", aspect="auto")
    plt.colorbar(im)

    plt.xticks(ticks=np.arange(len(feature_names)), labels=feature_names, rotation=45, ha='right')
    plt.yticks(ticks=np.arange(components), labels=[f"PC{i+1}" for i in range(components)])

    plt.title("PCA Loadings (Top Components)")
    plt.tight_layout()
    plt.show()



feature_cols = [
                "event_net_odds", "choice_ev", "choice_proba", "choice_fstar", 'choice_accuracy',
                "choice_decimal_odds", "choice_bet_sharpe", "choice_bet_sigma", 'choice_juice', 'sharpe_portfolio', 'portfolio_sigma', 'sigma_portfolio_scaled'
]

target_col = "event_net_odds"
target_col = 'f_event_net'
pipeline = run_linear_model_with_pca(
    df=df_kelly,                      # your original df_filt
    feature_cols=feature_cols,
    target_col=target_col,
    variance_threshold=0.95
)
plot_pca_loadings(pipeline["pca"], feature_cols)
